# CompoundT5 on 297k ORD reactions — the clean-base analogue of variant 5

Third point on the data-scaling curve for the ORD-free base (see
`12_train_reactant_compoundt5_57k.ipynb` for why `sagawa/CompoundT5` and why the vocabulary
repair is mandatory). 297,000 ORD reactions, 2.02x the 147k run.

**What this settles.** On `ReactionT5v2-retrosynthesis` the scaling curve went flat and then
backwards — variant 4 (147k) scored 47.3% ORD top-1 against variant 2's (57k) 50.3%, and variant
5 (297k) landed at 47.3% top-1 with top-5 core *falling* from 81.7% to 78.0%. `RESULTS.md`
records this as "further scaling without changing other factors gives no gain on this test set".
That base had 1.5M ORD reactions of pretraining behind it, so the added data was largely already
seen. On the clean base the first step of the same curve went the other way: 57k -> 147k improved
every metric on both test sets.

| | ORD exact top-1 | ORD exact top-5 | ORD core top-5 | eval_loss |
|---|---|---|---|---|
| CompoundT5 + 57k | 17.0% | 23.7% | 34.0% | 0.5678 |
| CompoundT5 + 147k | 19.7% | 28.3% | 39.7% | 0.4185 |
| ReactionT5 variant 2 (57k) | 50.3% | 73.0% | 79.7% | — |
| ReactionT5 variant 4 (147k) | 47.3% | 73.7% | 81.7% | — |
| ReactionT5 variant 5 (297k) | 47.3% | 70.0% | 78.0% | — |

Same recipe as both earlier runs — `--no-augment`, `lr=5e-4`, 3 epochs,
`torchrun --nproc_per_node=2` — so pool size stays the only difference across the three points.

**Data:** `kuzmenkooleh/retro-planner-ord-300k` (297,000 train + 3,000 val, the same pool variant
5 used). Variant 5 took ~8 h 02 min on T4 x2 for this size; the two evaluations add ~15 min.
Kaggle's per-session limit is 12 h, so this fits with headroom, but it is the single most
expensive run in the series.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All
(Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

# Kaggle has mounted datasets under two different layouts historically, so search
# rather than hard-code the path.
train_file = next(glob.iglob("/kaggle/input/**/reactants_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/reactants_val.jsonl", recursive=True))
for path in (train_file, val_file):
    print(path, sum(1 for _ in open(path)), "rows")
assert sum(1 for _ in open(train_file)) > 200000, "wrong dataset mounted -- expected the 297k pool"

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4  # same as the 57k and 147k runs, so pool size is the only difference
output_dir = "/kaggle/working/model1_compoundt5_300k"
time_budget_minutes = 560  # ~482 min training + headroom, under Kaggle's 12 h session cap

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate {learning_rate} \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The repair must have fired. If this line is absent the run trained on <unk>-corrupted
# targets and its numbers are meaningless.
!grep -E "new character token|Train examples" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"

# 0.5678 at 57k, 0.4185 at 147k. Where this lands says whether the curve is still moving.
state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[::max(1, len(points) // 10)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))

In [ ]:
import os
model_dir = f"{output_dir}/final"
assert os.path.isdir(model_dir), os.listdir(output_dir)

for tag, targets in [("ord", "data/v2_ord_eval_targets.json"),
                     ("uspto", "data/v2_uspto_eval_targets.json")]:
    !python scripts/models/run_reactiont5_topk.py \
        --input "{targets}" --t5-model "{model_dir}" \
        --num-beams 10 --device cuda \
        --output "/kaggle/working/compoundt5_300k_{tag}_topk.json"
    print(tag, "done")

In [ ]:
import json
for tag in ("ord", "uspto"):
    data = json.load(open(f"/kaggle/working/compoundt5_300k_{tag}_topk.json"))
    print("===", tag, "===")
    print(json.dumps(data["summary"], indent=2))